# Capítulo 5 · Actividad: ¿cuánto consume este auto?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/02_actividad_mpg_regresion.ipynb)

Entrena una **regresión** que prediga el rendimiento de un auto en **millas por galón** (`mpg`) y consigue el mejor modelo que puedas. Haz todo el proceso que vimos en clase; el preprocesamiento lo decides tú.

| Variable | Significado |
|---|---|
| `mpg` | **Target**: millas por galón (más alto = consume menos) |
| `cylinders` | Número de cilindros |
| `displacement` | Cilindrada del motor (pulgadas cúbicas) |
| `horsepower` | Potencia (caballos de fuerza) |
| `weight` | Peso (libras) |
| `acceleration` | Segundos para ir de 0 a 60 mph |
| `model_year` | Año del modelo (70 = 1970) |
| `origin` | Región de origen: `usa`, `europe` o `japan` |
| `name` | Nombre del auto |

Como mínimo, tu notebook debe incluir:

- Un **modelo lineal base**: coeficientes, métricas de error y $R^2$.
- Los mismos modelos que vimos en clase: regresión simple, lineal múltiple, Ridge, Lasso, polinomial y RANSAC.
- Una **tabla final** comparando los modelos con validación cruzada y test, y tus **conclusiones**: ¿qué modelo elegirías y por qué?

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`


## Importación de librerías


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, RANSACRegressor, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

palette = sns.color_palette("deep", 6)
sns.set_theme(style="whitegrid", palette=palette)
origin_palette = dict(zip(["usa", "europe", "japan"], palette[:3]))
correlation_cmap = sns.blend_palette([palette[1], "white", palette[0]], as_cmap=True)


## Carga de datos


In [ ]:
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(raiz / "data" / "mpg.csv")
df.head()


## Exploración inicial


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


## Valores faltantes


In [ ]:
pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
}).sort_values("nulos", ascending=False)


In [ ]:
df.loc[df["horsepower"].isna(), ["name", "origin", "cylinders", "weight", "mpg"]]


## Detección de duplicados


In [ ]:
df.duplicated().sum()


## Distribución de mpg


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="mpg", bins=20, color=palette[0], ax=axes[0])
sns.scatterplot(data=df, x="weight", y="mpg", color=palette[0], alpha=0.7, ax=axes[1])
plt.tight_layout()
plt.show()

## Variables categóricas y mpg


In [ ]:
df["origin"].value_counts().rename("autos")


In [ ]:
df["cylinders"].value_counts().sort_index().rename("autos")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(
    data=df, x="origin", y="mpg", hue="origin", palette=origin_palette,
    saturation=1, legend=False, ax=axes[0]
)
sns.boxplot(
    data=df, x="cylinders", y="mpg", hue="cylinders", palette=palette[:5],
    saturation=1, legend=False, ax=axes[1]
)
plt.tight_layout()
plt.show()


## Distribución de variables numéricas


In [ ]:
numeric_columns = ["displacement", "horsepower", "weight", "acceleration", "model_year"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for column, ax in zip(numeric_columns, axes.flat):
    sns.histplot(data=df, x=column, bins=20, color=palette[0], ax=ax)
axes.flat[-1].set_axis_off()
plt.tight_layout()
plt.show()


## Relaciones con mpg


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for column, ax in zip(numeric_columns, axes.flat):
    sns.scatterplot(data=df, x=column, y="mpg", color=palette[0], alpha=0.7, ax=ax)
axes.flat[-1].set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
sns.scatterplot(data=df, x="weight", y="mpg", hue="origin", palette=origin_palette, alpha=0.8)
plt.show()


## Matriz de correlación


In [ ]:
df_numeric = df.select_dtypes(include="number")
plt.figure(figsize=(8, 6))
sns.heatmap(df_numeric.corr(), annot=True, cmap=correlation_cmap, center=0, vmin=-1, vmax=1)
plt.tight_layout()
plt.show()


## Detección de valores atípicos


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for column, ax in zip(["displacement", "horsepower", "weight", "acceleration"], axes):
    sns.boxplot(data=df, y=column, color=palette[0], ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
continuous = ["displacement", "horsepower", "weight", "acceleration"]
q1 = df[continuous].quantile(0.25)
q3 = df[continuous].quantile(0.75)
iqr = q3 - q1
outliers = (df[continuous] < q1 - 1.5 * iqr) | (df[continuous] > q3 + 1.5 * iqr)
outliers.sum().rename("atipicos")


## Eliminación de name

`name` identifica cada auto y no entra como predictor.


In [ ]:
df = df.drop(columns="name")
df.head()


## Selección de variables predictoras y objetivo


In [ ]:
numeric_features = ["displacement", "horsepower", "weight", "acceleration", "model_year"]
categorical_features = ["origin", "cylinders"]

X = df[numeric_features + categorical_features]
y = df["mpg"]


## División en entrenamiento y prueba


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
X_train.shape, X_test.shape


## Atípicos de entrenamiento

Se conservan: los extremos de `horsepower` y `acceleration` son autos reales. RANSAC los trata en el ajuste.


In [ ]:
q1 = X_train[continuous].quantile(0.25)
q3 = X_train[continuous].quantile(0.75)
iqr = q3 - q1
outliers_train = (X_train[continuous] < q1 - 1.5 * iqr) | (X_train[continuous] > q3 + 1.5 * iqr)
outliers_train.sum().rename("atipicos_entrenamiento")


## Imputación, escalado y codificación


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        categories=[["europe", "japan", "usa"], [3, 4, 5, 6, 8]],
        drop="first", handle_unknown="ignore", sparse_output=False
    ))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## Modelo de control


In [ ]:
control_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("regression", LinearRegression())
])
control_model.fit(X_train, y_train)


## Coeficientes del control


In [ ]:
feature_names = control_model.named_steps["preprocessor"].get_feature_names_out()
print(f"Intercepto: {control_model.named_steps['regression'].intercept_:.3f}")
pd.Series(
    control_model.named_steps["regression"].coef_, index=feature_names, name="coeficiente"
).sort_values()


## Alternativa polinomial


In [ ]:
polynomial_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("polynomial", PolynomialFeatures(include_bias=False)),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("categorical", clone(categorical_pipeline), categorical_features)
])


## Modelos con hiperparámetros por defecto


In [ ]:
models = {
    "Control": control_model,
    "Simple": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("regression", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", Ridge())
    ]),
    "Lasso": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", Lasso())
    ]),
    "Polinomial": Pipeline([
        ("preprocessor", polynomial_preprocessor),
        ("regression", LinearRegression())
    ]),
    "RANSAC": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", RANSACRegressor(random_state=42))
    ])
}

model_columns = {}
for name in models:
    if name == "Simple":
        model_columns[name] = ["weight"]
    else:
        model_columns[name] = list(X.columns)

model_palette = dict(zip(models, palette))


## Comparación con los mismos 5 folds


In [ ]:
scoring = {"rmse": "neg_root_mean_squared_error", "r2": "r2"}
fold_rows = []
for name, estimator in models.items():
    scores = cross_validate(
        estimator, X_train[model_columns[name]], y_train,
        cv=cv, scoring=scoring, error_score="raise"
    )
    for fold in range(5):
        fold_rows.append({
            "modelo": name,
            "fold": fold + 1,
            "RMSE": -scores["test_rmse"][fold],
            "R2": scores["test_r2"][fold]
        })

cv_results = pd.DataFrame(fold_rows)
cv_results.pivot(index="fold", columns="modelo", values="RMSE")


In [ ]:
comparison_cv = cv_results.groupby("modelo").agg(
    RMSE_CV=("RMSE", "mean"), RMSE_std=("RMSE", "std"), R2_CV=("R2", "mean")
).sort_values("RMSE_CV")
comparison_cv


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for metric, ax in zip(["RMSE", "R2"], axes):
    sns.barplot(
        data=cv_results, x=metric, y="modelo", hue="modelo", palette=model_palette,
        saturation=1, errorbar="sd", capsize=0.15, legend=False, ax=ax
    )
    ax.set_ylabel("")
axes[0].set_title("RMSE en mpg: menor es mejor")
axes[1].set_title("R²: mayor es mejor")
plt.tight_layout()
plt.show()


## Optimización del modelo con menor RMSE


In [ ]:
best_name = comparison_cv["RMSE_CV"].idxmin()
param_grids = {
    "Control": {},
    "Simple": {},
    "Ridge": {"regression__alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso": {"regression__alpha": [0.001, 0.01, 0.1, 1]},
    "Polinomial": {"preprocessor__numeric__polynomial__degree": [1, 2, 3]},
    "RANSAC": {"regression__residual_threshold": [None, 3, 5]}
}
search = GridSearchCV(
    models[best_name], param_grids[best_name], scoring=scoring,
    refit="rmse", cv=cv, error_score="raise"
)
search.fit(X_train[model_columns[best_name]], y_train)
optimized_model = search.best_estimator_
print(best_name)
print(f"RMSE CV por defecto: {comparison_cv.loc[best_name, 'RMSE_CV']:.3f}")
print(f"RMSE CV tras el grid: {-search.best_score_:.3f}")
search.best_params_


## Evaluación en el conjunto de prueba


In [ ]:
final_models = {}
for name, estimator in models.items():
    final_models[name] = clone(estimator)
final_models["Optimizado"] = optimized_model

final_columns = {}
for name in models:
    final_columns[name] = model_columns[name]
final_columns["Optimizado"] = model_columns[best_name]

final_rows = []
predictions = {}
for name, estimator in final_models.items():
    columns = final_columns[name]
    if name != "Optimizado":
        estimator.fit(X_train[columns], y_train)
    pred = estimator.predict(X_test[columns])
    predictions[name] = pred
    if name == "Optimizado":
        rmse_cv = -search.best_score_
    else:
        rmse_cv = comparison_cv.loc[name, "RMSE_CV"]
    final_rows.append({
        "modelo": name,
        "RMSE_CV": rmse_cv,
        "MAE_prueba": mean_absolute_error(y_test, pred),
        "RMSE_prueba": root_mean_squared_error(y_test, pred),
        "R2_prueba": r2_score(y_test, pred)
    })

comparison = pd.DataFrame(final_rows).set_index("modelo")
comparison


## Control frente al optimizado


In [ ]:
comparison_plot = comparison.loc[["Control", "Optimizado"]].reset_index()
comparison_palette = {"Control": palette[0], "Optimizado": palette[1]}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for metric, ax in zip(["MAE_prueba", "RMSE_prueba", "R2_prueba"], axes):
    sns.barplot(
        data=comparison_plot, x="modelo", y=metric, hue="modelo",
        palette=comparison_palette, saturation=1, legend=False, errorbar=None, ax=ax
    )
    ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
    ax.set_ylim(0, comparison_plot[metric].max() * 1.15)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


## Predicciones


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
limits = [y_test.min(), y_test.max()]
for ax, name in zip(axes, ["Control", "Optimizado"]):
    pred = predictions[name]
    sns.scatterplot(x=y_test, y=pred, color=comparison_palette[name], ax=ax)
    sns.lineplot(x=limits, y=limits, color="gray", linestyle="--", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("MPG real")
    ax.set_ylabel("MPG predicho")
plt.tight_layout()
plt.show()
